In [ ]:
!pip install netCDF4 numpy xarray scipy pyproj cartopy
!pip install git+https://github.com/NCAR/wrf-python

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
from netCDF4 import Dataset
from wrf import getvar, latlon_coords, to_np
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import geopandas as gpd
import numpy as np
import pandas as pd
import cartopy.mpl.ticker as cticker
import matplotlib.ticker as mticker

# =====================================
# CONFIGURATION
# =====================================
base_path = "/content/drive/MyDrive/WRF"
experiments = ["2005", "2025"]
timeidx = 7

# AOI Sidoarjo
lon_min, lon_max = 112.45, 112.90
lat_min, lat_max = -7.60, -7.30

geojson = "/content/drive/MyDrive/WRF/2005/gadm41_IDN_2.json"
boundary = gpd.read_file(geojson)

def setup_plot(ax, title):
    ax.add_geometries(boundary.geometry, ccrs.PlateCarree(), edgecolor='black', facecolor='none', linewidth=0.8)
    ax.set_extent([lon_min, lon_max, lat_min, lat_max], crs=ccrs.PlateCarree())
    gl = ax.gridlines(draw_labels=True, linewidth=0.4, linestyle='--', alpha=0.5)
    gl.top_labels = gl.right_labels = False
    gl.xformatter = cticker.LongitudeFormatter()
    gl.yformatter = cticker.LatitudeFormatter()
    ax.set_title(title, fontsize=10, fontweight='bold')

# =====================================
# DATA PROCESSING & PLOTTING
# =====================================
t2_dict = {}

for exp in experiments:
    print(f"Processing T2 for experiment: {exp}")
    ncfile = Dataset(f"{base_path}/{exp}/wrfout_d04_")

    # Get T2 and convert to Celsius
    t2_var = getvar(ncfile, "T2", timeidx=timeidx)
    data_t2 = to_np(t2_var) - 273.15
    t2_dict[exp] = data_t2

    lats, lons = latlon_coords(t2_var)

    fig, ax = plt.subplots(figsize=(8, 7), subplot_kw={'projection': ccrs.PlateCarree()})
    cs = ax.contourf(lons, lats, data_t2,
                     levels=np.arange(24, 36, 1),
                     cmap="jet", extend="both")
    setup_plot(ax, f"Temperature at 2m (T2) Tahun {exp} (°C)")
    plt.colorbar(cs, orientation='horizontal', pad=0.08, label="°C")
    plt.show()
    ncfile.close()

# =====================================
# STATISTICAL COMPARISON
# =====================================
stats_t2 = pd.DataFrame({
    "Metric": ["Mean", "Std Dev", "Min", "Max"],
    "2005 (°C)": [np.nanmean(t2_dict["2005"]), np.nanstd(t2_dict["2005"]), np.nanmin(t2_dict["2005"]), np.nanmax(t2_dict["2005"])],
    "2025 (°C)": [np.nanmean(t2_dict["2025"]), np.nanstd(t2_dict["2025"]), np.nanmin(t2_dict["2025"]), np.nanmax(t2_dict["2025"])]
})

stats_t2["Selisih (2025-2005)"] = stats_t2["2025 (°C)"] - stats_t2["2005 (°C)"]

print("\n--- Perbandingan Nilai Statistik Temperature at 2m (T2) ---")
display(stats_t2.round(4))

# Menampilkan nilai selisih rata-rata
mean_diff_t2 = stats_t2.loc[stats_t2['Metric'] == 'Mean', 'Selisih (2025-2005)'].values[0]
print(f"\nSelisih Rata-rata T2: {mean_diff_t2:.4f} °C")

In [ ]:
from netCDF4 import Dataset
from wrf import getvar, latlon_coords, to_np
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import geopandas as gpd
import numpy as np
import pandas as pd
import cartopy.mpl.ticker as cticker
import matplotlib.ticker as mticker

# =====================================
# CONFIGURATION
# =====================================
base_path = "/content/drive/MyDrive/WRF"
experiments = ["2005", "2025"]
timeidx = 7

# AOI Sidoarjo
lon_min, lon_max = 112.45, 112.90
lat_min, lat_max = -7.60, -7.30

geojson = "/content/drive/MyDrive/WRF/2005/gadm41_IDN_2.json"
boundary = gpd.read_file(geojson)

def setup_plot(ax, title):
    ax.add_geometries(boundary.geometry, ccrs.PlateCarree(), edgecolor='black', facecolor='none', linewidth=0.8)
    ax.set_extent([lon_min, lon_max, lat_min, lat_max], crs=ccrs.PlateCarree())
    gl = ax.gridlines(draw_labels=True, linewidth=0.4, linestyle='--', alpha=0.5)
    gl.top_labels = gl.right_labels = False
    gl.xformatter = cticker.LongitudeFormatter()
    gl.yformatter = cticker.LatitudeFormatter()
    ax.set_title(title, fontsize=10, fontweight='bold')

# =====================================
# DATA PROCESSING & PLOTTING
# =====================================
q2_dict = {}

for exp in experiments:
    print(f"Processing Specific Humidity (Q2) for experiment: {exp}")
    ncfile = Dataset(f"{base_path}/{exp}/wrfout_d04_")

    # Get Q2 and convert to g/kg
    q2_var = getvar(ncfile, "Q2", timeidx=timeidx)
    data_q2 = to_np(q2_var) * 1000
    q2_dict[exp] = data_q2

    lats, lons = latlon_coords(q2_var)

    fig, ax = plt.subplots(figsize=(8, 7), subplot_kw={'projection': ccrs.PlateCarree()})
    cs = ax.contourf(lons, lats, data_q2,
                     levels=np.linspace(np.nanpercentile(data_q2, 2), np.nanpercentile(data_q2, 98), 20),
                     cmap="YlGnBu", extend="both")
    setup_plot(ax, f"Specific Humidity (Q2) Tahun {exp} (g/kg)")
    plt.colorbar(cs, orientation='horizontal', pad=0.08, label="g kg$^{-1}$")
    plt.show()
    ncfile.close()

# =====================================
# STATISTICAL COMPARISON
# =====================================
stats_q2 = pd.DataFrame({
    "Metric": ["Mean", "Std Dev", "Min", "Max"],
    "2005 (g/kg)": [np.nanmean(q2_dict["2005"]), np.nanstd(q2_dict["2005"]), np.nanmin(q2_dict["2005"]), np.nanmax(q2_dict["2005"])],
    "2025 (g/kg)": [np.nanmean(q2_dict["2025"]), np.nanstd(q2_dict["2025"]), np.nanmin(q2_dict["2025"]), np.nanmax(q2_dict["2025"])]
})

stats_q2["Selisih (2025-2005)"] = stats_q2["2025 (g/kg)"] - stats_q2["2005 (g/kg)"]

print("\n--- Perbandingan Nilai Statistik Specific Humidity (Q2) ---")
display(stats_q2.round(4))

# Menampilkan nilai selisih rata-rata
mean_diff_q2 = stats_q2.loc[stats_q2['Metric'] == 'Mean', 'Selisih (2025-2005)'].values[0]
print(f"\nSelisih Rata-rata Q2: {mean_diff_q2:.4f} g/kg")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import geopandas as gpd
from netCDF4 import Dataset
from wrf import getvar, latlon_coords, to_np
import cartopy.mpl.ticker as cticker
import matplotlib.ticker as mticker

# =====================================
# 1. Konfigurasi Input
# =====================================
base_path = "/content/drive/MyDrive/WRF"
experiments = ["2005", "2025"]
timeidx = 7
geojson = "/content/drive/MyDrive/WRF/2005/gadm41_IDN_2.json"

# AOI Sidoarjo
lon_min, lon_max = 112.45, 112.90
lat_min, lat_max = -7.60, -7.30

boundary = gpd.read_file(geojson)

def setup_plot(ax, title):
    ax.add_geometries(boundary.geometry, ccrs.PlateCarree(), edgecolor='black', facecolor='none', linewidth=0.8)
    ax.set_extent([lon_min, lon_max, lat_min, lat_max], crs=ccrs.PlateCarree())
    gl = ax.gridlines(draw_labels=True, linewidth=0.4, linestyle='--', alpha=0.5)
    gl.top_labels = gl.right_labels = False
    gl.xformatter = cticker.LongitudeFormatter()
    gl.yformatter = cticker.LatitudeFormatter()
    ax.set_title(title, fontsize=10, fontweight='bold')

def calculate_rh(nc, t_idx):
    t2_var = getvar(nc, "T2", timeidx=t_idx)
    psfc = getvar(nc, "PSFC", timeidx=t_idx)
    q2 = getvar(nc, "Q2", timeidx=t_idx)
    T = to_np(t2_var)
    P = to_np(psfc) / 100
    q = to_np(q2)
    e = (q * P) / (0.622 + 0.378 * q)
    Tc = T - 273.15
    es = 6.112 * np.exp((17.67 * Tc) / (Tc + 243.5))
    rh = np.clip(100 * e / es, 0, 100)
    return rh, t2_var

# =====================================
# 2. Visualisasi RH & Pengumpulan Data
# =====================================
rh_data = {}

for exp in experiments:
    ncfile = Dataset(f"{base_path}/{exp}/wrfout_d04_")
    rh_val, t2_ref = calculate_rh(ncfile, timeidx)
    rh_data[exp] = rh_val
    lats, lons = latlon_coords(t2_ref)

    fig, ax = plt.subplots(figsize=(8, 7), subplot_kw={'projection': ccrs.PlateCarree()})
    cs = ax.contourf(lons, lats, rh_val, levels=np.arange(40, 101, 5), cmap='BrBG', extend='both')
    setup_plot(ax, f"Relative Humidity (%) Tahun {exp}")
    plt.colorbar(cs, orientation='horizontal', pad=0.08, label='%')
    plt.show()
    ncfile.close()

# =====================================
# 3. Statistik Perbandingan Selisih RH
# =====================================
stats_rh = pd.DataFrame({
    "Metric": ["Mean", "Std Dev", "Min", "Max"],
    "2005 (%)": [np.nanmean(rh_data["2005"]), np.nanstd(rh_data["2005"]), np.nanmin(rh_data["2005"]), np.nanmax(rh_data["2005"])],
    "2025 (%)": [np.nanmean(rh_data["2025"]), np.nanstd(rh_data["2025"]), np.nanmin(rh_data["2025"]), np.nanmax(rh_data["2025"])]
})

stats_rh["Selisih (2025-2005)"] = stats_rh["2025 (%)"] - stats_rh["2005 (%)"]

print("\n--- Perbandingan Nilai Statistik Relative Humidity (RH) ---")
display(stats_rh.round(3))

# Menambahkan summary selisih rata-rata
mean_diff_rh = stats_rh.loc[stats_rh['Metric'] == 'Mean', 'Selisih (2025-2005)'].values[0]
print(f"\nSelisih Rata-rata RH: {mean_diff_rh:.3f} % ")

In [ ]:
from netCDF4 import Dataset
from wrf import getvar, latlon_coords, to_np
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import geopandas as gpd
import numpy as np
import pandas as pd
import cartopy.mpl.ticker as cticker
import matplotlib.ticker as mticker

# =====================================
# CONFIGURATION
# =====================================
base_path = "/content/drive/MyDrive/WRF"
exp_05 = "2005"
exp_25 = "2025"
timeidx = 7

# AOI Sidoarjo
lon_min, lon_max = 112.45, 112.90
lat_min, lat_max = -7.60, -7.30

geojson = "/content/drive/MyDrive/WRF/2005/gadm41_IDN_2.json"
boundary = gpd.read_file(geojson)

def setup_plot(ax, title):
    ax.add_geometries(boundary.geometry, ccrs.PlateCarree(), edgecolor='black', facecolor='none', linewidth=0.8)
    ax.set_extent([lon_min, lon_max, lat_min, lat_max], crs=ccrs.PlateCarree())
    gl = ax.gridlines(draw_labels=True, linewidth=0.4, linestyle='--', alpha=0.5)
    gl.top_labels = gl.right_labels = False
    gl.xformatter = cticker.LongitudeFormatter()
    gl.yformatter = cticker.LatitudeFormatter()
    ax.set_title(title, fontsize=10, fontweight='bold')

# =====================================
# LOAD DATA
# =====================================
nc_05 = Dataset(f"{base_path}/{exp_05}/wrfout_d04_")
nc_25 = Dataset(f"{base_path}/{exp_25}/wrfout_d04_")

# Get coordinates and HFX data
lats, lons = latlon_coords(getvar(nc_05, "T2", timeidx=timeidx))
hfx_05 = to_np(getvar(nc_05, "HFX", timeidx=timeidx))
hfx_25 = to_np(getvar(nc_25, "HFX", timeidx=timeidx))

# =====================================
# PLOTTING
# =====================================
# 1. Plot HFX 2005
fig1, ax1 = plt.subplots(figsize=(7, 6), subplot_kw={'projection': ccrs.PlateCarree()})
cs1 = ax1.contourf(lons, lats, hfx_05, levels=np.arange(-50, 401, 25), cmap="hot", extend="both")
setup_plot(ax1, f"HFX {exp_05} (W/m2)")
plt.colorbar(cs1, orientation='horizontal', pad=0.08, label="W m$^{-2}$")
plt.show()

# 2. Plot HFX 2025
fig2, ax2 = plt.subplots(figsize=(7, 6), subplot_kw={'projection': ccrs.PlateCarree()})
cs2 = ax2.contourf(lons, lats, hfx_25, levels=np.arange(-50, 401, 25), cmap="hot", extend="both")
setup_plot(ax2, f"HFX {exp_25} (W/m2)")
plt.colorbar(cs2, orientation='horizontal', pad=0.08, label="W m$^{-2}$")
plt.show()

# Calculate Statistics
stats_data = {
    "Metric": ["Mean", "Std Dev", "Min", "Max"],
    "2005 (W/m2)": [
        np.nanmean(hfx_05), np.nanstd(hfx_05), np.nanmin(hfx_05), np.nanmax(hfx_05)
    ],
    "2025 (W/m2)": [
        np.nanmean(hfx_25), np.nanstd(hfx_25), np.nanmin(hfx_25), np.nanmax(hfx_25)
    ]
}

df_stats = pd.DataFrame(stats_data)

# Calculate Numeric Difference (2025 - 2005)
df_stats["Selisih (2025-2005)"] = df_stats["2025 (W/m2)"] - df_stats["2005 (W/m2)"]

print("Perbandingan Statistik HFX (Sensible Heat Flux):")
display(df_stats.round(3))

# Print summary of mean difference
mean_diff = df_stats.loc[df_stats['Metric'] == 'Mean', 'Selisih (2025-2005)'].values[0]
print(f"\nSelisih Rata-rata HFX: {mean_diff:.3f} W/m2")

# Close datasets once
nc_05.close()
nc_25.close()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import geopandas as gpd
from netCDF4 import Dataset
from wrf import getvar, latlon_coords, to_np, destagger
import cartopy.mpl.ticker as cticker
import matplotlib.ticker as mticker

# =====================================
# 1. Konfigurasi Input
# =====================================
base_path = "/content/drive/MyDrive/WRF"
experiments = ["2005", "2025"]
timeidx = 7
geojson = "/content/drive/MyDrive/WRF/2005/gadm41_IDN_2.json"

# AOI Sidoarjo
lon_min, lon_max = 112.45, 112.90
lat_min, lat_max = -7.60, -7.30

boundary = gpd.read_file(geojson)

def setup_plot(ax, title):
    ax.add_geometries(boundary.geometry, ccrs.PlateCarree(), edgecolor='black', facecolor='none', linewidth=0.8)
    ax.set_extent([lon_min, lon_max, lat_min, lat_max], crs=ccrs.PlateCarree())
    gl = ax.gridlines(draw_labels=True, linewidth=0.4, linestyle='--', alpha=0.5)
    gl.top_labels = gl.right_labels = False
    gl.xformatter = cticker.LongitudeFormatter()
    gl.yformatter = cticker.LatitudeFormatter()
    ax.set_title(title, fontsize=10, fontweight='bold')

# =====================================
# 2. Visualisasi Moisture Flux & Pengumpulan Data
# =====================================
mflux_dict = {}

for exp in experiments:
    ncfile = Dataset(f"{base_path}/{exp}/wrfout_d04_")

    # Ambil data kelembaban spesifik dan angin
    q = getvar(ncfile, "QVAPOR", timeidx=timeidx)
    u = getvar(ncfile, "U", timeidx=timeidx)
    v = getvar(ncfile, "V", timeidx=timeidx)
    lats, lons = latlon_coords(q)

    # Destagger ke grid massa dan pilih level terbawah
    u_mass = destagger(u, 2)[0, :, :]
    v_mass = destagger(v, 1)[0, :, :]
    q_bot = q[0, :, :]

    # Hitung Magnitudo Moisture Flux
    mflux = np.sqrt((to_np(q_bot * u_mass))**2 + (to_np(q_bot * v_mass))**2)
    mflux_dict[exp] = mflux

    fig, ax = plt.subplots(figsize=(8, 7), subplot_kw={'projection': ccrs.PlateCarree()})
    cs = ax.contourf(lons, lats, mflux, levels=np.linspace(0, 0.15, 16), cmap='YlGnBu', extend='max')
    setup_plot(ax, f"Moisture Flux Tahun {exp}")
    plt.colorbar(cs, orientation='horizontal', pad=0.08, label=r'kg kg$^{-1}$ m s$^{-1}$')
    plt.show()
    ncfile.close()

# =====================================
# 3. Statistik Perbandingan Moisture Flux
# =====================================
stats_mflux = pd.DataFrame({
    "Metric": ["Mean", "Std Dev", "Min", "Max"],
    "2005 (kg/kg.m/s)": [
        np.nanmean(mflux_dict["2005"]), np.nanstd(mflux_dict["2005"]),
        np.nanmin(mflux_dict["2005"]), np.nanmax(mflux_dict["2005"])
    ],
    "2025 (kg/kg.m/s)": [
        np.nanmean(mflux_dict["2025"]), np.nanstd(mflux_dict["2025"]),
        np.nanmin(mflux_dict["2025"]), np.nanmax(mflux_dict["2025"])
    ]
})

# Hitung Selisih (2025 - 2005)
stats_mflux["Selisih (2025-2005)"] = stats_mflux["2025 (kg/kg.m/s)"] - stats_mflux["2005 (kg/kg.m/s)"]

print("\n--- Perbandingan Statistik Moisture Flux ---")
display(stats_mflux.round(6))

# Print summary selisih rata-rata
mean_diff = stats_mflux.loc[stats_mflux['Metric'] == 'Mean', 'Selisih (2025-2005)'].values[0]
print(f"\nSelisih Rata-rata Moisture Flux: {mean_diff:.6f} kg kg-1 m s-1")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import geopandas as gpd
from netCDF4 import Dataset
from wrf import getvar, latlon_coords, to_np
import cartopy.mpl.ticker as cticker
import matplotlib.ticker as mticker

# =====================================
# 1. Konfigurasi Input
# =====================================
base_path = "/content/drive/MyDrive/WRF"
experiments = ["2005", "2025"]
timeidx = 7
geojson = "/content/drive/MyDrive/WRF/2005/gadm41_IDN_2.json"

# AOI Sidoarjo
lon_min, lon_max = 112.45, 112.90
lat_min, lat_max = -7.60, -7.30

boundary = gpd.read_file(geojson)

def setup_plot(ax, title):
    ax.add_geometries(boundary.geometry, ccrs.PlateCarree(), edgecolor='black', facecolor='none', linewidth=0.8)
    ax.set_extent([lon_min, lon_max, lat_min, lat_max], crs=ccrs.PlateCarree())
    gl = ax.gridlines(draw_labels=True, linewidth=0.4, linestyle='--', alpha=0.5)
    gl.top_labels = gl.right_labels = False
    gl.xformatter = cticker.LongitudeFormatter()
    gl.yformatter = cticker.LatitudeFormatter()
    ax.set_title(title, fontsize=10, fontweight='bold')

# =====================================
# 2. Visualisasi PBLH & Pengumpulan Data
# =====================================
pblh_dict = {}

for exp in experiments:
    ncfile = Dataset(f"{base_path}/{exp}/wrfout_d04_")

    # Ambil data PBLH dan koordinat
    pblh = getvar(ncfile, "PBLH", timeidx=timeidx)
    lats, lons = latlon_coords(pblh)
    data_pblh = to_np(pblh)
    pblh_dict[exp] = data_pblh

    fig, ax = plt.subplots(figsize=(8, 7), subplot_kw={'projection': ccrs.PlateCarree()})
    cs = ax.contourf(lons, lats, data_pblh, levels=np.arange(0, 2501, 100), cmap='Spectral_r', extend='max')
    setup_plot(ax, f"PBL Height (PBLH) Tahun {exp}")
    plt.colorbar(cs, orientation='horizontal', pad=0.08, label='Meter (m)')
    plt.show()
    ncfile.close()

# =====================================
# 3. Statistik Perbandingan PBLH
# =====================================
stats_pblh = pd.DataFrame({
    "Metric": ["Mean", "Std Dev", "Min", "Max"],
    "2005 (m)": [
        np.nanmean(pblh_dict["2005"]), np.nanstd(pblh_dict["2005"]),
        np.nanmin(pblh_dict["2005"]), np.nanmax(pblh_dict["2005"])
    ],
    "2025 (m)": [
        np.nanmean(pblh_dict["2025"]), np.nanstd(pblh_dict["2025"]),
        np.nanmin(pblh_dict["2025"]), np.nanmax(pblh_dict["2025"])
    ]
})

# Hitung Selisih (2025 - 2005)
stats_pblh["Selisih (2025-2005)"] = stats_pblh["2025 (m)"] - stats_pblh["2005 (m)"]

print("\n--- Perbandingan Nilai Statistik PBL Height (PBLH) ---")
display(stats_pblh.round(3))

# Print summary selisih rata-rata
mean_diff = stats_pblh.loc[stats_pblh['Metric'] == 'Mean', 'Selisih (2025-2005)'].values[0]
print(f"\nSelisih Rata-rata PBLH: {mean_diff:.3f} meter")